# 🎯 Fitts' Law Experiment Analysis

Analyze head-controlled pointing data from multiple participants.

## 🚀 How to Use:
1. Run Cell 1 (Setup)
2. Run Cell 2 (Upload) - Upload fitts-student-data.zip
3. Run remaining cells to see results!

In [ ]:
# 1️⃣ Setup - Install packages
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import warnings
import os
warnings.filterwarnings('ignore')

sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)
print("✅ Packages loaded!")

In [ ]:
# 2️⃣ Upload Data
from google.colab import files
import zipfile

print("📁 Upload your fitts-student-data.zip file:")
print("   (Click 'Choose Files' and select the zip file)")
print("")
uploaded = files.upload()

# Extract zip files
for filename in uploaded.keys():
    if filename.endswith('.zip'):
        print(f"\n📦 Extracting {filename}...")
        with zipfile.ZipFile(filename, 'r') as zip_ref:
            zip_ref.extractall('.')
        print(f"✅ Extracted!")

# List what we have
print("\n📂 Files in current directory:")
for item in os.listdir('.'):
    if os.path.isdir(item) and not item.startswith('.'):
        print(f"   📁 {item}/")
        # Show subdirectories
        try:
            subdirs = [d for d in os.listdir(item) if os.path.isdir(os.path.join(item, d)) and not d.startswith('.')]
            if subdirs:
                print(f"      Contains {len(subdirs)} folders")
        except:
            pass

In [ ]:
# 3️⃣ Load Data (searches all subdirectories)
import glob

# Search for CSV files recursively
raw_files = glob.glob('**/fitts-experiment-raw-data*.csv', recursive=True)
results_files = glob.glob('**/fitts-experiment-results*.csv', recursive=True)

print(f"🔍 Searching for CSV files...")
print(f"   Found {len(raw_files)} raw data files")
print(f"   Found {len(results_files)} results files")
print("")

if len(raw_files) == 0:
    print("❌ No CSV files found!")
    print("")
    print("💡 Try this:")
    print("   1. Make sure you uploaded fitts-student-data.zip")
    print("   2. Or upload CSV files directly (without zip)")
    print("")
    print("📂 Current directory contents:")
    !ls -la
else:
    # Load raw data
    raw_list = []
    for i, file in enumerate(raw_files, 1):
        try:
            df = pd.read_csv(file)
            # Extract participant name from path
            parts = file.split('/')
            participant = parts[-2] if len(parts) > 1 else f'P{i}'
            df['ParticipantID'] = participant
            raw_list.append(df)
            print(f"   ✅ {participant}: {len(df)} trials")
        except Exception as e:
            print(f"   ⚠️  Error loading {file}: {e}")

    if raw_list:
        raw_data = pd.concat(raw_list, ignore_index=True)
        print(f"\n✅ Total: {len(raw_data)} trials from {raw_data['ParticipantID'].nunique()} participants")
    else:
        print("❌ No data could be loaded")
        raw_data = None

    # Load results if available
    if results_files:
        results_list = []
        for file in results_files:
            try:
                df = pd.read_csv(file)
                parts = file.split('/')
                participant = parts[-2] if len(parts) > 1 else 'Unknown'
                df['ParticipantID'] = participant
                results_list.append(df)
            except:
                pass
        
        if results_list:
            results_data = pd.concat(results_list, ignore_index=True)
            print(f"✅ Loaded {len(results_data)} aggregated results")
        else:
            results_data = None
    else:
        results_data = None
        print("ℹ️  No pre-aggregated results, will calculate from raw data")

In [ ]:
# 4️⃣ Calculate Metrics (if needed)
if raw_data is not None:
    if results_data is None:
        print("📊 Calculating Fitts metrics from raw data...")
        results = []
        
        grouping_cols = ['ParticipantID', 'FilterType', 'TargetSize', 'Amplitude']
        grouping_cols = [col for col in grouping_cols if col in raw_data.columns]
        
        for group_keys, group in raw_data.groupby(grouping_cols):
            Ae = group['ActualAmplitude'].mean()
            # ISO 9241-9 directional projection: project endpoint deviation onto movement direction
            if 'Direction' in group.columns and 'TargetX' in group.columns:
                theta_rad = np.radians(group['Direction'].astype(float))
                dx = group['SelectionX'] - group['TargetX']
                dy = group['SelectionY'] - group['TargetY']
                projections = dx * np.cos(theta_rad) + dy * np.sin(theta_rad)
                We = 4.133 * projections.std()
            else:
                dx = group['SelectionX'] - group['TargetX']
                dy = group['SelectionY'] - group['TargetY']
                distances = np.sqrt(dx**2 + dy**2)
                We = 4.133 * distances.std()
            IDe = np.log2(Ae / We + 1) if We > 0 else 0
            MeanMT = group['MovementTime'].mean()
            TP = IDe / MeanMT if MeanMT > 0 else 0
            
            result = dict(zip(grouping_cols, group_keys))
            result.update({
                'N': len(group),
                'MeanMT': MeanMT,
                'Ae': Ae,
                'We': We,
                'IDe': IDe,
                'TP': TP
            })
            results.append(result)
        
        results_data = pd.DataFrame(results)
        print(f"✅ Calculated {len(results)} conditions")
    else:
        print("✅ Using pre-calculated results")

    print("\n📊 Results preview:")
    display(results_data.head(10))
else:
    print("❌ No raw data available. Please upload data first.")

In [ ]:
# 5️⃣ Summary Statistics
if results_data is not None:
    print("="*80)
    print("📊 SUMMARY STATISTICS")
    print("="*80)
    print(f"\n👥 Participants: {results_data['ParticipantID'].nunique()}")
    print(f"📊 Total Trials: {len(raw_data)}")
    print(f"\n🎯 Overall Performance:")
    print(f"   Throughput: {results_data['TP'].mean():.3f} ± {results_data['TP'].std():.3f} bits/s")
    print(f"   Movement Time: {results_data['MeanMT'].mean():.3f} ± {results_data['MeanMT'].std():.3f} s")

    if 'FilterType' in results_data.columns:
        print(f"\n🔧 Performance by Filter:")
        filter_stats = results_data.groupby('FilterType').agg({
            'TP': ['mean', 'std', 'min', 'max'],
            'MeanMT': ['mean', 'std']
        }).round(3)
        display(filter_stats)

In [ ]:
# 6️⃣ Statistical Tests
if results_data is not None and 'FilterType' in results_data.columns:
    print("="*80)
    print("📊 STATISTICAL TESTS")
    print("="*80)

    filters = results_data['FilterType'].unique()
    if len(filters) == 2:
        f1, f2 = filters
        
        # Get throughput by participant
        tp1 = results_data[results_data['FilterType'] == f1].groupby('ParticipantID')['TP'].mean()
        tp2 = results_data[results_data['FilterType'] == f2].groupby('ParticipantID')['TP'].mean()
        
        print(f"\n📈 Throughput Comparison:")
        print(f"   {f1}: {tp1.mean():.3f} ± {tp1.std():.3f} bits/s")
        print(f"   {f2}: {tp2.mean():.3f} ± {tp2.std():.3f} bits/s")
        
        # Paired t-test
        common = set(tp1.index) & set(tp2.index)
        if len(common) > 1:
            tp1_vals = [tp1[p] for p in common]
            tp2_vals = [tp2[p] for p in common]
            t_stat, p_val = stats.ttest_rel(tp1_vals, tp2_vals)
            
            print(f"\n   Paired t-test: t({len(common)-1})={t_stat:.3f}, p={p_val:.4f}")
            
            if p_val < 0.05:
                better = f1 if np.mean(tp1_vals) > np.mean(tp2_vals) else f2
                worse = f2 if better == f1 else f1
                improvement = abs((np.mean(tp1_vals) - np.mean(tp2_vals)) / min(np.mean(tp1_vals), np.mean(tp2_vals)) * 100)
                print(f"   ✅ SIGNIFICANT DIFFERENCE (p < 0.05)")
                print(f"   🏆 {better} is {improvement:.1f}% better than {worse}")
            else:
                print(f"   ❌ No significant difference (p >= 0.05)")
            
            # Cohen's d
            diff = np.array(tp1_vals) - np.array(tp2_vals)
            cohens_d = np.mean(diff) / np.std(diff, ddof=1)
            print(f"   Effect size (Cohen's d): {abs(cohens_d):.3f}")
            
            if abs(cohens_d) < 0.2:
                print(f"      → Small effect")
            elif abs(cohens_d) < 0.5:
                print(f"      → Medium effect")
            else:
                print(f"      → Large effect")

In [ ]:
# 7️⃣ Graph 1: Throughput by Filter
if results_data is not None and 'FilterType' in results_data.columns:
    plt.figure(figsize=(10, 6))
    sns.boxplot(data=results_data, x='FilterType', y='TP', palette='Set2')
    sns.swarmplot(data=results_data, x='FilterType', y='TP', color='black', alpha=0.3, size=4)
    plt.title('Throughput by Filter Type', fontsize=16, fontweight='bold')
    plt.ylabel('Throughput (bits/s)', fontsize=12)
    plt.xlabel('Filter Type', fontsize=12)
    plt.grid(axis='y', alpha=0.3)
    plt.tight_layout()
    plt.savefig('throughput_by_filter.png', dpi=300, bbox_inches='tight')
    plt.show()
    print("💾 Saved: throughput_by_filter.png")

In [ ]:
# 8️⃣ Graph 2: Movement Time by Filter
if results_data is not None and 'FilterType' in results_data.columns:
    plt.figure(figsize=(10, 6))
    sns.boxplot(data=results_data, x='FilterType', y='MeanMT', palette='Set2')
    sns.swarmplot(data=results_data, x='FilterType', y='MeanMT', color='black', alpha=0.3, size=4)
    plt.title('Movement Time by Filter Type', fontsize=16, fontweight='bold')
    plt.ylabel('Movement Time (s)', fontsize=12)
    plt.xlabel('Filter Type', fontsize=12)
    plt.grid(axis='y', alpha=0.3)
    plt.tight_layout()
    plt.savefig('movement_time_by_filter.png', dpi=300, bbox_inches='tight')
    plt.show()
    print("💾 Saved: movement_time_by_filter.png")

In [ ]:
# 9️⃣ Graph 3: Participant Performance
if results_data is not None and 'FilterType' in results_data.columns:
    participant_tp = results_data.groupby(['ParticipantID', 'FilterType'])['TP'].mean().reset_index()

    plt.figure(figsize=(14, 6))
    sns.barplot(data=participant_tp, x='ParticipantID', y='TP', hue='FilterType', palette='Set2')
    plt.title('Individual Participant Throughput', fontsize=16, fontweight='bold')
    plt.ylabel('Throughput (bits/s)', fontsize=12)
    plt.xlabel('Participant', fontsize=12)
    plt.xticks(rotation=45, ha='right')
    plt.legend(title='Filter')
    plt.grid(axis='y', alpha=0.3)
    plt.tight_layout()
    plt.savefig('participant_throughput.png', dpi=300, bbox_inches='tight')
    plt.show()
    print("💾 Saved: participant_throughput.png")

In [ ]:
# 🔟 Graph 4: Fitts' Law Regression
if results_data is not None and 'FilterType' in results_data.columns and 'IDe' in results_data.columns:
    plt.figure(figsize=(10, 6))

    for ftype in results_data['FilterType'].unique():
        data = results_data[results_data['FilterType'] == ftype]
        plt.scatter(data['IDe'], data['MeanMT'], label=ftype, alpha=0.6, s=50)
        
        # Regression line
        z = np.polyfit(data['IDe'], data['MeanMT'], 1)
        p = np.poly1d(z)
        x_line = np.linspace(data['IDe'].min(), data['IDe'].max(), 100)
        plt.plot(x_line, p(x_line), '--', linewidth=2)

    plt.xlabel('Index of Difficulty (bits)', fontsize=12)
    plt.ylabel('Movement Time (s)', fontsize=12)
    plt.title("Fitts' Law: MT = a + b × ID", fontsize=16, fontweight='bold')
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig('fitts_law_regression.png', dpi=300, bbox_inches='tight')
    plt.show()
    print("💾 Saved: fitts_law_regression.png")

In [ ]:
# 1️⃣1️⃣ Graph 5: Performance by Target Size
if results_data is not None and 'TargetSize' in results_data.columns and 'FilterType' in results_data.columns:
    plt.figure(figsize=(12, 6))
    sns.barplot(data=results_data, x='TargetSize', y='TP', hue='FilterType', palette='Set2')
    plt.title('Throughput by Target Size', fontsize=16, fontweight='bold')
    plt.ylabel('Throughput (bits/s)', fontsize=12)
    plt.xlabel('Target Size (pixels)', fontsize=12)
    plt.legend(title='Filter')
    plt.grid(axis='y', alpha=0.3)
    plt.tight_layout()
    plt.savefig('throughput_by_size.png', dpi=300, bbox_inches='tight')
    plt.show()
    print("💾 Saved: throughput_by_size.png")

In [ ]:
# 1️⃣2️⃣ Export CSV Files
if results_data is not None:
    # Summary by filter
    if 'FilterType' in results_data.columns:
        summary_filter = results_data.groupby('FilterType').agg({
            'TP': ['mean', 'std', 'min', 'max'],
            'MeanMT': ['mean', 'std', 'min', 'max']
        }).round(4)
        summary_filter.to_csv('summary_by_filter.csv')
        print("💾 Saved: summary_by_filter.csv")

    # Summary by participant
    if 'FilterType' in results_data.columns:
        summary_participant = results_data.groupby(['ParticipantID', 'FilterType']).agg({
            'TP': 'mean',
            'MeanMT': 'mean'
        }).round(4)
        summary_participant.to_csv('summary_by_participant.csv')
        print("💾 Saved: summary_by_participant.csv")

    # All results
    results_data.to_csv('all_results.csv', index=False)
    print("💾 Saved: all_results.csv")

    print("\n✅ Done! Download files from the Files panel (📁 icon on left sidebar)")
    print("\n📊 Summary of what you have:")
    print("   • 5 graphs (PNG, 300 DPI)")
    print("   • 3 CSV summary tables")
    print("   • Statistical test results above")